# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library for Python.

### Dataset Source
The dataset source is specified by a Croissant schema URL, which describes its structure and enables automated access and processing.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and inspect its description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {', '.join(metadata.keywords) if metadata.keywords else 'N/A'}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Each entity (record set, field, column) has a unique `@id`. We'll use these for precise operations.

In [ ]:
# List all record sets in the dataset by @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}  | name: {rs.get('name', '<no name>')}")

# For illustration, print field info for each record set
for rs in record_sets:
    print(f"\nFields for Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        # 'field' might be a reference or dict - normalize
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
        print(f"    - Field @id: {field_id}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for detailed analysis.

We'll use the precise `@id` of each record set as seen above for data loading.

In [ ]:
# Collect all record set @id values for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
print("Loading all record sets into DataFrames by @id...")
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded: {record_set_id} ({len(df)} records)")
        else:
            print(f"Skipped (no records): {record_set_id}")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

# Show columns of the first loaded record set
if dataframes:
    sample_record_set_id = list(dataframes.keys())[0]
    print(f"\nSample columns for {sample_record_set_id}:")
    print(dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head())
else:
    print("No record sets contained data.")

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate common EDA operations, including filtering numeric fields, normalization, and grouping.
All operations reference data by field or column `@id`, per best practices.

In [ ]:
# Choose a sample record set and numeric field
if dataframes:
    sample_rs_id = list(dataframes.keys())[0]
    df = dataframes[sample_rs_id]
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.to_list()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # reference by @id/column name
        print(f"Using numeric field (by @id): {numeric_field_id}")

        # Example: filter records where field > threshold
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by another field (categorical), if one exists
        group_candidates = df.select_dtypes(include=['object', 'category']).columns.to_list()
        group_field_id = None
        for col in group_candidates:
            if len(df[col].unique()) < 20:
                group_field_id = col
                break
        if group_field_id:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields available in this record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Let's plot distributions and relationships between fields in the selected record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[sample_rs_id]
    if numeric_candidates:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()

    # If a grouping field was found, visualize group differences
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=30)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to programmatically load, access, and analyze the FAIR² dataset of adoption predictors in rangeland management practices.

- All entities are referenced by their `@id` fields for clarity and automation.
- We inspected record set structures, extracted data, and conducted basic exploratory data analysis and visualization.

For deeper analysis, consult the dataset's record sets and schema for additional field insights, and explore more advanced EDA or modeling suitable to your research or application.